# Full-data GRU: торговля от депозита $1000

Ноутбук проверяет сохраненную модель `gru_full_direction_best.pth`, обученную на полном ряде без фильтрации только по событиям. Для демонстрационного графика используется режим одной открытой позиции за раз, стартовый капитал $1000 и риск 1% на сделку.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path(r"C:/Projects/robot")

sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import pandas as pd
import plotly.graph_objects as go

from src import config
from src.connector.data_fetcher import load_all_price_data
from src.models.sequence_models import load_model_with_config
from src.strategy.backtest import (
    apply_risk_sizing,
    build_trades,
    calculate_risk_equity_curve,
    calculate_risk_trade_metrics,
)
from src.strategy.signal_generator import generate_signal_history

pd.set_option("display.max_columns", None)

In [2]:
INITIAL_CAPITAL = 1000.0
RISK_PER_TRADE_PCT = 1.0

# Подобранная демонстрационная конфигурация: сделок меньше, график читается лучше.
CONFIDENCE_THRESHOLD = 0.65
TP_THRESHOLD = 0.0008  # 8 пунктов для EURUSD
SL_THRESHOLD = 0.0004  # 4 пункта для EURUSD
COST_PER_TRADE = 0.00002  # примерно 0.2 пункта на сделку
HORIZON = 8  # 8 свечей по 15 минут = 2 часа
TEST_START = "2024-01-01"

print(f"Стартовый капитал: ${INITIAL_CAPITAL:,.2f}")
print(f"TP: {TP_THRESHOLD * 10000:.1f} pips, SL: {SL_THRESHOLD * 10000:.1f} pips")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD:.2f}")

Стартовый капитал: $1,000.00
TP: 8.0 pips, SL: 4.0 pips
Confidence threshold: 0.65


In [3]:
price_df, loaded_files = load_all_price_data(config.DATA_DIR)

test_start = pd.Timestamp(TEST_START)
if price_df.index.tz is not None:
    test_start = test_start.tz_localize(price_df.index.tz)

max_rows = int((price_df.index >= test_start).sum())

model = load_model_with_config(config.GRU_FULL_MODEL_PATH, config.GRU_FULL_CONFIG_PATH)
scaler = joblib.load(config.GRU_FULL_SCALER_PATH)

signals = generate_signal_history(
    price_df,
    threshold=CONFIDENCE_THRESHOLD,
    model_path=config.GRU_FULL_MODEL_PATH,
    scaler_path=config.GRU_FULL_SCALER_PATH,
    model_type="gru",
    require_event=False,
    max_rows=max_rows,
    model=model,
    scaler=scaler,
)
signals = signals[signals["time"] >= test_start].copy()

print(f"Загружено CSV-файлов: {len(loaded_files)}")
print(f"Период теста: {signals['time'].min()} -> {signals['time'].max()}")
print(f"Всего свечей в тесте: {len(signals):,}")
print(f"Торговых сигналов модели: {(signals['decision'] != 'NO TRADE').sum():,}")

Загружено CSV-файлов: 45
Период теста: 2024-01-01 22:00:00+00:00 -> 2026-03-26 16:30:00+00:00
Всего свечей в тесте: 55,093
Торговых сигналов модели: 478


In [4]:
raw_trades = build_trades(
    signals,
    price_df,
    horizon=HORIZON,
    tp_threshold=TP_THRESHOLD,
    sl_threshold=SL_THRESHOLD,
    cost_per_trade=COST_PER_TRADE,
)


def keep_one_position_at_a_time(trades: pd.DataFrame) -> pd.DataFrame:
    """Оставляет следующую сделку только после выхода из предыдущей."""
    if trades.empty:
        return trades.copy()

    entry_col = trades.columns[0]
    selected_rows = []
    next_allowed_time = pd.Timestamp.min
    if price_df.index.tz is not None:
        next_allowed_time = next_allowed_time.tz_localize(price_df.index.tz)

    for _, trade in trades.sort_values(entry_col).iterrows():
        entry_time = pd.Timestamp(trade[entry_col])
        exit_time = pd.Timestamp(trade["Exit time"])
        if entry_time >= next_allowed_time:
            selected_rows.append(trade)
            next_allowed_time = exit_time

    return pd.DataFrame(selected_rows).reset_index(drop=True)


trades = keep_one_position_at_a_time(raw_trades)
trades = apply_risk_sizing(
    trades,
    initial_capital=INITIAL_CAPITAL,
    risk_per_trade_pct=RISK_PER_TRADE_PCT,
    sl_threshold=SL_THRESHOLD,
)

metrics = calculate_risk_trade_metrics(trades, initial_capital=INITIAL_CAPITAL)
equity_curve = calculate_risk_equity_curve(trades, initial_capital=INITIAL_CAPITAL)

summary = pd.DataFrame(
    {
        "Показатель": [
            "Стартовый капитал, $",
            "Финальный капитал, $",
            "Доходность, %",
            "Сделок после фильтра одной позиции",
            "Win rate, %",
            "Profit factor",
            "Max drawdown, %",
            "Средняя сделка, $",
        ],
        "Значение": [
            INITIAL_CAPITAL,
            metrics["Current Capital"],
            metrics["total_return"] * 100,
            metrics["Trades"],
            metrics["winrate"] * 100,
            metrics["profit_factor"],
            metrics["max_drawdown"] * 100,
            metrics["Average Trade Money"],
        ],
    }
)
summary

,Показатель,Значение
0,"Стартовый капитал, $",1000.000000
1,"Финальный капитал, $",1409.055675
2,"Доходность, %",40.905567
3,Сделок после фильтра одной позиции,266.000000
4,"Win rate, %",48.496241
5,Profit factor,1.273410
6,"Max drawdown, %",-12.162211
7,"Средняя сделка, $",1.537803


In [5]:
threshold_rows = []
thresholds = [0.53, 0.55, 0.57, 0.60, 0.62, 0.65, 0.67, 0.70]

for threshold in thresholds:
    threshold_signals = signals.copy()
    threshold_signals["decision"] = "NO TRADE"
    buy_mask = (threshold_signals["prediction"] == "UP") & (threshold_signals["confidence"] >= threshold)
    sell_mask = (threshold_signals["prediction"] == "DOWN") & (threshold_signals["confidence"] >= threshold)
    threshold_signals.loc[buy_mask, "decision"] = "BUY"
    threshold_signals.loc[sell_mask, "decision"] = "SELL"

    threshold_raw_trades = build_trades(
        threshold_signals,
        price_df,
        horizon=HORIZON,
        tp_threshold=TP_THRESHOLD,
        sl_threshold=SL_THRESHOLD,
        cost_per_trade=COST_PER_TRADE,
    )
    threshold_trades = keep_one_position_at_a_time(threshold_raw_trades)
    threshold_trades = apply_risk_sizing(
        threshold_trades,
        initial_capital=INITIAL_CAPITAL,
        risk_per_trade_pct=RISK_PER_TRADE_PCT,
        sl_threshold=SL_THRESHOLD,
    )
    threshold_metrics = calculate_risk_trade_metrics(threshold_trades, initial_capital=INITIAL_CAPITAL)

    threshold_rows.append(
        {
            "threshold": threshold,
            "signals": int((threshold_signals["decision"] != "NO TRADE").sum()),
            "trades_one_position": int(threshold_metrics["Trades"]),
            "final_capital": threshold_metrics["Current Capital"],
            "return_pct": threshold_metrics["total_return"] * 100,
            "winrate_pct": threshold_metrics["winrate"] * 100,
            "profit_factor": threshold_metrics["profit_factor"],
            "max_drawdown_pct": threshold_metrics["max_drawdown"] * 100,
        }
    )

threshold_table = pd.DataFrame(threshold_rows)
threshold_table


,threshold,signals,trades_one_position,final_capital,return_pct,winrate_pct,profit_factor,max_drawdown_pct
0,0.53,21068,9201,267.275736,-73.272426,37.702424,0.966333,-89.178884
1,0.55,11372,5212,4505.545274,350.554527,40.195702,1.055206,-62.378943
2,0.57,6344,3009,7260.372446,626.037245,42.472582,1.134083,-47.831404
3,0.60,2617,1314,3704.104818,270.410482,44.368341,1.224298,-28.855346
4,0.62,1373,739,1584.424765,58.442476,44.384303,1.124389,-32.712282
5,0.65,478,266,1409.055675,40.905567,48.496241,1.273410,-12.162211
6,0.67,203,113,1157.872101,15.787210,46.902655,1.275349,-8.412641
7,0.70,53,33,1183.980039,18.398004,63.636364,2.477060,-3.117041


In [6]:
entry_col = equity_curve.columns[0]

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=equity_curve[entry_col],
        y=equity_curve["Equity"],
        mode="lines",
        name="Капитал",
        line={"color": "#2563eb", "width": 2.5},
    )
)
fig.add_hline(
    y=INITIAL_CAPITAL,
    line_dash="dash",
    line_color="#6b7280",
    annotation_text="Стартовый капитал",
    annotation_position="top left",
)

annotation = (
    f"TP/SL: {TP_THRESHOLD * 10000:.0f}/{SL_THRESHOLD * 10000:.0f} pips<br>"
    f"Risk: {RISK_PER_TRADE_PCT:.1f}%<br>"
    f"Trades: {metrics['Trades']}<br>"
    f"Final: ${metrics['Current Capital']:,.2f}<br>"
    f"Max DD: {metrics['max_drawdown'] * 100:.1f}%"
)
fig.add_annotation(
    xref="paper",
    yref="paper",
    x=0.02,
    y=0.98,
    text=annotation,
    showarrow=False,
    align="left",
    bgcolor="white",
    bordercolor="#d1d5db",
    borderwidth=1,
)
fig.update_layout(
    title="Full-data GRU: equity curve от депозита $1000",
    xaxis_title="Дата",
    yaxis_title="Капитал, $",
    template="plotly_white",
    width=1100,
    height=560,
)

output_path = PROJECT_ROOT / "Pictures" / "full_data_gru_equity_1000.html"
fig.write_html(output_path)
print(f"Интерактивный график сохранен: {output_path}")
fig.show()

Интерактивный график сохранен: C:\Projects\robot\Pictures\full_data_gru_equity_1000.html


In [7]:
trades.head(10)

,Дата,Инструмент,Сигнал,confidence,Цена входа,Цена выхода,Результат,Exit time,Exit reason,Gross PnL,Cost,PnL,PnL pips,Risk Amount,Lot Size,PnL Money,Equity
0,2024-01-07 23:15:00+00:00,EURUSD,BUY,0.6566,1.09356,1.09436,WIN,2024-01-08 00:00:00+00:00,TP,0.00080,0.00002,0.00078,7.8,10.000000,0.250000,19.500000,1019.500000
1,2024-01-08 21:15:00+00:00,EURUSD,BUY,0.6669,1.09515,1.09518,WIN,2024-01-08 23:15:00+00:00,HORIZON,0.00003,0.00002,0.00001,0.1,10.195000,0.254875,0.254875,1019.754875
2,2024-01-15 18:45:00+00:00,EURUSD,SELL,0.6521,1.09528,1.09517,WIN,2024-01-15 20:45:00+00:00,HORIZON,0.00011,0.00002,0.00009,0.9,10.197549,0.254939,2.294448,1022.049323
3,2024-01-24 20:45:00+00:00,EURUSD,BUY,0.6592,1.08800,1.08851,WIN,2024-01-24 22:45:00+00:00,HORIZON,0.00051,0.00002,0.00049,4.9,10.220493,0.255512,12.520104,1034.569428
4,2024-01-26 21:30:00+00:00,EURUSD,BUY,0.6609,1.08523,1.08483,LOSS,2024-01-28 22:00:00+00:00,SL,-0.00040,0.00002,-0.00042,-4.2,10.345694,0.258642,-10.862979,1023.706449
5,2024-01-28 22:00:00+00:00,EURUSD,BUY,0.6694,1.08443,1.08424,LOSS,2024-01-29 00:00:00+00:00,HORIZON,-0.00019,0.00002,-0.00021,-2.1,10.237064,0.255927,-5.374459,1018.331990
6,2024-02-07 21:45:00+00:00,EURUSD,BUY,0.6710,1.07704,1.07754,WIN,2024-02-07 23:45:00+00:00,HORIZON,0.00050,0.00002,0.00048,4.8,10.183320,0.254583,12.219984,1030.551974
7,2024-02-12 21:45:00+00:00,EURUSD,BUY,0.6547,1.07706,1.07717,WIN,2024-02-12 23:45:00+00:00,HORIZON,0.00011,0.00002,0.00009,0.9,10.305520,0.257638,2.318742,1032.870716
8,2024-02-14 21:45:00+00:00,EURUSD,BUY,0.6723,1.07250,1.07319,WIN,2024-02-14 23:45:00+00:00,HORIZON,0.00069,0.00002,0.00067,6.7,10.328707,0.258218,17.300584,1050.171300
9,2024-02-19 17:30:00+00:00,EURUSD,SELL,0.6570,1.07757,1.07797,LOSS,2024-02-19 18:00:00+00:00,SL,-0.00040,0.00002,-0.00042,-4.2,10.501713,0.262543,-11.026799,1039.144501
